# 🤿 Notebook 3 — FathomNet: Caption Inference with OpenAI/ChatGPT



## 0. Install Dependencies


In [ ]:
!pip install openai pillow tqdm pandas -q


## 1. Setup & Config


In [ ]:
import os, sqlite3, shutil, time, base64, io
from pathlib import Path
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# ── Load API Key từ Kaggle Secrets ──────────────────────────────────────────
secrets = UserSecretsClient()
OPENAI_API_KEY = secrets.get_secret('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

# ── Paths ───────────────────────────────────────────────────────────────────
INPUT_DB  = Path('/kaggle/input/fathomnet-underwater-db/fathomnet.db')
INPUT_IMG = Path('/kaggle/input/fathomnet-underwater-db/images')

if not INPUT_IMG.exists():
    possible_images = list(Path('/kaggle/input').rglob('images'))
    if possible_images:
        INPUT_IMG = possible_images[0]

OUT_DIR   = Path('/kaggle/working/fathomnet_captioned_openai')
OUT_DB    = OUT_DIR / 'fathomnet_openai_captions.db'
OUT_CSV   = OUT_DIR / 'image_text_pairs.csv'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Copy DB
if not OUT_DB.exists() and INPUT_DB.exists():
    shutil.copy2(INPUT_DB, OUT_DB)
    print(f"Copied DB -> {OUT_DB}")

# ── OpenAI Model Config ──────────────────────────────────────────────────────
MODEL_NAME = 'gpt-4o-mini'
DELAY_BETWEEN_REQUESTS = 0.05  

print(f"Model: {MODEL_NAME}")
print("Setup complete!")


## 2. Prompt Builder


In [ ]:
def build_prompt(species: str, depth, temp) -> str:
    context_parts = []
    if species and species.strip() and species.strip().lower() not in ['unknown', '']:
        context_parts.append(f"phylum/species: {species.strip()}")
    if depth and not pd.isna(depth):
        context_parts.append(f"depth: {float(depth):.0f}m")
    if temp and not pd.isna(temp):
        context_parts.append(f"water temp: {float(temp):.1f}°C")

    ctx = ', '.join(context_parts)
    
    prompt = "This is a cropped underwater photo of a marine organism."
    if ctx:
        prompt += f" Known metadata: {ctx}."
    prompt += """
Describe what you see in 2-3 sentences (max 70 words).
Focus on:
1. The organism's appearance, color, texture, shape, and notable features.
2. The visual quality of the photo: explicitly describe if the image contains heavy digital noise, graininess, black/white dots (salt-and-pepper noise), camera artifacts, or marine snow.
Be specific and factual. Do NOT mention the word 'caption'. Start directly with the description."""
    return prompt


## 3. Inference Loop


In [ ]:
def encode_image(img_path: Path, max_size=768) -> str:
    img = Image.open(img_path).convert('RGB')
    if max(img.size) > max_size:
        ratio = max_size / max(img.size)
        new_size = (int(img.width * ratio), int(img.height * ratio))
        img = img.resize(new_size, Image.LANCZOS)
    
    buffer = io.BytesIO()
    # Nén chất lượng 85 để giảm thiểu lượng byte gửi lên OpenAI (tiết kiệm token)
    img.save(buffer, format="JPEG", quality=85)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

def caption_with_openai(img_path: Path, prompt: str, max_retries: int = 5) -> str:
    base64_img = encode_image(img_path)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_img}",
                                    "detail": "low"  # Detail: 'low' giúp tối ưu cực hạn chi phí, chỉ tốn $0.0001/ảnh
                                }
                            }
                        ]
                    }
                ],
                max_tokens=100,
                temperature=0.3
            )
            return response.choices[0].message.content.strip()
            
        except Exception as e:
            err_str = str(e).lower()
            print(f"\n[Attempt {attempt+1}/{max_retries} FAILED] Error: {str(e)}")
            
            if 'rate limit' in err_str or '429' in err_str:
                wait = 5 * (attempt + 1)
                time.sleep(wait)
            else:
                time.sleep(5)
                
    raise RuntimeError(f"🛑 API KHÔNG THỂ KẾT NỐI SAU {max_retries} LẦN THỬ!")

conn = sqlite3.connect(OUT_DB)
df_images = pd.read_sql('SELECT * FROM images', conn)

try:
    done_ids = set(pd.read_sql('SELECT image_id FROM image_text_pairs', conn)['image_id'])
    print(f"Already captioned: {len(done_ids)}")
except Exception:
    done_ids = set()

df_todo = df_images[~df_images['image_id'].isin(done_ids)].copy()
print(f"Remaining to caption: {len(df_todo)}")

BATCH_SIZE = 50
results = []
errors = []

for row in tqdm(df_todo.itertuples(), total=len(df_todo), desc='OpenAI Captioning'):
    img_path = INPUT_IMG / row.filename
    if not img_path.exists():
        img_path = INPUT_IMG / 'api_images' / Path(row.filename).name
    if not img_path.exists():
        errors.append(row.image_id)
        continue
    
    prompt = build_prompt(getattr(row, 'species_clean', ''), getattr(row, 'depth_m', None), getattr(row, 'temperature_c', None))
    
    try:
        caption = caption_with_openai(img_path, prompt)
        if caption:
            results.append((row.image_id, caption, prompt, MODEL_NAME))
        else:
            errors.append(row.image_id)
    except Exception:
        errors.append(row.image_id)
        
    if len(results) >= BATCH_SIZE:
        conn.executemany('INSERT OR IGNORE INTO image_text_pairs (image_id, caption, prompt_used, model_name) VALUES (?,?,?,?)', results)
        conn.commit()
        results = []
    
    time.sleep(DELAY_BETWEEN_REQUESTS)

if results:
    conn.executemany('INSERT OR IGNORE INTO image_text_pairs (image_id, caption, prompt_used, model_name) VALUES (?,?,?,?)', results)
    conn.commit()

conn.close()
print(f"\n✅ Inference complete!")
print(f"   Errors / skipped: {len(errors)}")


## 4. Export CSV & check


In [ ]:
conn = sqlite3.connect(OUT_DB)
df_final = pd.read_sql("""
    SELECT 
        i.filename        AS image_path,
        i.species_clean   AS species,
        p.caption,
        p.model_name
    FROM image_text_pairs p
    JOIN images i ON p.image_id = i.image_id
""", conn)
conn.close()

df_final.to_csv(OUT_CSV, index=False)
print(f"Saved CSV: {OUT_CSV} with {len(df_final)} rows")
print("\n=== 5 Mẫu ngẫu nhiên ===")
for _, r in df_final.sample(min(5, len(df_final))).iterrows():
    print(f"[{r['species']}] {r['caption']}\n")
